In [1]:
import csv
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
base = "Dino"

threshold = 3.5

#file_GT = "Input Data/snr4_mid_benchmark.csv"
#file_Dino = "Input Data/snr4_mid_centroids.csv"

#ile_GT = "Input Data/snr4_high_benchmark.csv"
#file_Dino = "Input Data/snr4_high_centroids.csv"

#file_GT = "Input Data/snr7_mid_benchmark.csv"
#file_Dino = "Input Data/snr_7_mid_centroids.csv"

#file_GT = "Input Data/snr7_high_benchmark.csv"
#file_Dino = "Input Data/snr_7_high_centroids.csv"

file_GT = "Input Data/snr2_mid_benchmark.csv"
file_Dino = "Input Data/snr2_mid_centroids.csv"

In [3]:
df_GT = pd.read_csv(file_GT)
df_GT = df_GT[["ID", "t", "x", "y", "z"]]
display(df_GT)
M_GT = df_GT.to_numpy()

df_Dino = pd.read_csv(file_Dino, names = ["timepoint", "particle_id","z", "y", "x"],skiprows=1)
df_Dino = df_Dino[["particle_id", "timepoint", "x", "y", "z"]].sort_values(by=["timepoint", "particle_id"],ascending=[True,True])
display(df_Dino)
M_Dino = df_Dino.to_numpy()

if base == "GT":
    comp_list = np.zeros((len(M_GT),13), dtype=object)
    M_x = M_GT
    M_y = M_Dino
    sec = "Dino"
if base == "Dino":
    comp_list = np.zeros((len(M_Dino),13), dtype=object)
    M_x = M_Dino
    M_y = M_GT
    sec = "GT"

,ID,t,x,y,z
0,105,0,179.182,175.364,0.091
1,105,1,178.909,174.727,0.273
2,105,2,178.727,174.000,0.455
3,105,3,179.000,173.545,0.091
4,105,4,177.909,173.727,0.091
...,...,...,...,...,...
46973,2883,99,248.636,438.182,3.636
46974,2884,96,507.727,346.455,5.091
46975,2884,97,506.909,346.909,5.091
46976,2884,98,506.000,347.818,5.000


,particle_id,timepoint,x,y,z
0,0,0,254.766770,258.425450,4.011640
1,1,0,86.813140,1.585845,1.380683
2,2,0,98.458176,0.501558,0.817274
3,3,0,120.999756,0.000000,0.000000
4,4,0,166.485630,0.425288,1.772925
...,...,...,...,...,...
326719,3026,99,370.653600,510.509640,7.451407
326720,3027,99,415.203120,509.089300,7.878043
326721,3028,99,493.635960,510.690600,7.999994
326722,3029,99,501.926900,510.535030,6.929743


In [4]:
t_vec = defaultdict(list)
multi_match_list = []
for vec_y in M_y:
    t_vec[vec_y[1]].append(vec_y)

for t_val in t_vec:
    t_vec[t_val] = np.array(t_vec[t_val])

for i, vec_x in enumerate(M_x):
    t_val = vec_x[1]
    y_group = t_vec[t_val]
    diffs = y_group[:, 2:5] - vec_x[2:5]
    dists = np.linalg.norm(diffs, axis=1)
    min_idx = np.argmin(dists)
    min_dist = dists[min_idx]
    best_vec_y = y_group[min_idx]
    comp_list[i, 0:5] = vec_x
    comp_list[i, 5:10] = best_vec_y
    comp_list[i, 10] = min_dist

    below = np.where(dists < threshold)[0]
    if len(below) > 0:
        y_ids = ",".join(str(int(x)) for x in y_group[below, 0])
        dists_str = ",".join(f"{dists[j]:.2f}" for j in below)
    else:
        y_ids = ""
        dists_str = ""
    comp_list[i, 11] = y_ids
    comp_list[i, 12] = dists_str

multi_match_array = np.array(multi_match_list, dtype=object)

In [5]:
df_comp = pd.DataFrame(comp_list, columns = [f"ID ({base})", "t",f"x ({base})", f"y ({base})", 
                                             f"z ({base})", f"ID ({sec})", "t_ig", 
                                             f"x ({sec})", f"y ({sec})", f"z ({sec})", 
                                             "Distance", f"Multi ID ({sec})", f"Multi Distance ({sec})"])

df_comp = df_comp.drop("t_ig", axis=1)
df_sorted = df_comp.sort_values(by=[f"ID ({base})","t"], ascending=[True, True])

In [6]:
df_sorted.to_csv(f"GT Base Output/DetectionComparison.csv", index=False)
#df_sorted.to_csv(f"DC_SNR_2_mid.csv", index=False)

In [7]:
df_sorted[df_sorted["Distance"]<3.5]

,ID (Dino),t,x (Dino),y (Dino),z (Dino),ID (GT),x (GT),y (GT),z (GT),Distance,Multi ID (GT),Multi Distance (GT)
59318,0.0,18.0,246.78958,251.52039,4.041723,664.0,248.182,251.909,3.636,1.501487,664,1.50
145928,0.0,45.0,259.59296,259.52972,3.978508,1428.0,260.0,257.273,5.545,2.777114,1428,2.78
233859,0.0,72.0,255.44669,252.54485,4.064002,2264.0,255.0,255.636,5.0,3.260496,2264,3.26
244104,0.0,75.0,255.86179,255.55493,4.038587,2264.0,254.818,256.909,5.182,2.056793,2264,2.06
283514,0.0,87.0,256.20004,250.08131,4.081157,2045.0,256.545,249.636,5.455,1.484838,2045,1.48
...,...,...,...,...,...,...,...,...,...,...,...,...
124773,3509.0,37.0,420.19485,499.51254,6.785539,791.0,422.909,499.455,4.818,3.35278,791,3.35
111684,3510.0,33.0,250.94557,509.7373,6.437928,981.0,251.455,510.818,9.091,2.909677,981,2.91
323624,3531.0,98.0,437.6563,494.21698,6.803915,1720.0,438.273,494.818,9.0,2.358884,1720,2.36
323633,3540.0,98.0,56.424545,498.52274,6.005061,2613.0,57.182,497.909,7.091,1.459342,2613,1.46
